In [1]:
# rumus learning curve
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve

def plot_my_learning_curve(model, X, y, title="Learning Curve"):
    plt.figure(figsize=(8, 5))
    
    # Hitung learning curve
    train_sizes, train_scores, test_scores = learning_curve(
        model, X, y, cv=5, scoring='r2', n_jobs=-1, 
        train_sizes=np.linspace(0.1, 1.0, 5), random_state=42
    )
    
    # Hitung rata-rata dan standar deviasi
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    test_mean = np.mean(test_scores, axis=1)
    test_std = np.std(test_scores, axis=1)
    
    # Gambar grafik
    plt.plot(train_sizes, train_mean, 'o-', color="r", label="Training score")
    plt.plot(train_sizes, test_mean, 'o-', color="g", label="Cross-validation score")
    
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color="r")
    plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color="g")
    
    plt.title(title)
    plt.xlabel("Training Examples")
    plt.ylabel("R2 Score")
    plt.legend(loc="best")
    plt.grid(True)
    plt.show()

# CARA PAKAI (Jalankan setelah kamu load X dan y Supplies):
# plot_my_learning_curve(best_supplies_model, X_full, y_full, title="Learning Curve - Extra Trees Supplies")

In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Impor semua algoritma yang dibutuhkan berdasarkan tabel PyCaret-mu
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor, AdaBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import ElasticNet, Lars, Lasso, LinearRegression

# =========================================================================
# 1. PREPARASI DATA MASTER MINGGUAN
# =========================================================================
# Pastikan nama file sesuai dengan dataset mingguanmu
df_master = pd.read_csv(r'D:/Sem 4/PBL/data/processed/data_proses_mingguan.csv')

# Fungsi evaluasi custom MAPE
def calculate_mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) if np.sum(mask) > 0 else 0.0

# =========================================================================
# 2. DICTIONARY MAPPING (Kunci Otomatisasi)
# Kita pasangkan setiap Sub-Kategori dengan Model TOP 1 hasil PyCaret-mu
# =========================================================================
top_models_mapping = {
    "Accessories": ("GBR", GradientBoostingRegressor(random_state=42)),
    "Appliances":  ("Random_Forest", RandomForestRegressor(random_state=42)),
    "Art":         ("GBR", GradientBoostingRegressor(random_state=42)),
    "Binders":     ("Elastic_Net", ElasticNet(random_state=42)),
    "Bookcases":   ("Extra_Trees", ExtraTreesRegressor(random_state=42)),
    "Chairs":      ("Extra_Trees", ExtraTreesRegressor(random_state=42)),
    "Envelopes":   ("GBR", GradientBoostingRegressor(random_state=42)),
    "Fasteners":   ("Random_Forest", RandomForestRegressor(random_state=42)),
    "Furnishings": ("Random_Forest", RandomForestRegressor(random_state=42)),
    "Labels":      ("Random_Forest", RandomForestRegressor(random_state=42)),
    "Paper":       ("Random_Forest", RandomForestRegressor(random_state=42)),
    "Phones":      ("Random_Forest", RandomForestRegressor(random_state=42)),
    "Storage":     ("Extra_Trees", ExtraTreesRegressor(random_state=42)),
    "Supplies":    ("Lars", Lars()),
    "Tables":      ("Random_Forest", RandomForestRegressor(random_state=42))
}

# =========================================================================
# 3. SETTING MLFLOW EXPERIMENT
# =========================================================================
mlflow.set_tracking_uri("file:///D:/Sem 4/PBL/model-pycaret/mlruns")
mlflow.set_experiment("Weekly_SubCategory_Manual_Modeling")

print("🚀 Memulai otomatisasi training untuk 15 Sub-Kategori...")

# =========================================================================
# 4. LOOPING AUTOMATION PIPELINE
# =========================================================================
for sub_cat, (model_name, model_obj) in top_models_mapping.items():
    print(f"\n-------------------------------------------------------")
    print(f"[PROCESSING] Sub-Kategori: {sub_cat} menggunakan {model_name}")
    
    # a. Filter data spesifik untuk sub-kategori saat ini
    df_sub = df_master[df_master['Sub-Category'] == sub_cat].copy()
    
    # Jika data kosong atau terlalu sedikit, lewati untuk menghindari error
    if df_sub.shape[0] < 10:
        print(f"⚠️ Data untuk {sub_cat} terlalu sedikit ({df_sub.shape[0]} baris), dilewati.")
        continue
        
    # b. Pisahkan Fitur dan Target
    # Kita buang 'Sub-Category' karena datanya sudah homogen/terfilter
    X = df_sub.drop(columns=['Order Date', 'Sales', 'Profit', 'Quantity', 'Sub-Category'], errors='ignore')
    y = df_sub['Quantity']
    
    # c. Split Data (80:20)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # d. Jalankan Tracking MLflow dengan NAMA RUN DINAMIS
    run_identity = f"Weekly_{sub_cat}_{model_name}"
    with mlflow.start_run(run_name=run_identity):
        
        # Train Model
        model_obj.fit(X_train, y_train)
        
        # Prediksi
        y_pred = model_obj.predict(X_test)
        
        # Hitung Metrik
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        mape = calculate_mape(y_test, y_pred)
        
        # Log ke MLflow
        mlflow.log_param("sub_category", sub_cat)
        mlflow.log_params(model_obj.get_params())
        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("RMSE", rmse)
        mlflow.log_metric("R2", r2)
        mlflow.log_metric("MAPE", mape)
        
        # Simpan Model Artifact ke MLflow Registry
        mlflow.sklearn.log_model(model_obj, artifact_path=f"model_{sub_cat.lower()}")
        
        # Cetak hasil ke terminal notebook
        print(f"✨ {run_identity} Berhasil Tercatat!")
        print(f"   |-- Data Train/Test : {X_train.shape[0]} / {X_test.shape[0]} baris")
        print(f"   |-- MAE  : {mae:.4f} | RMSE : {rmse:.4f} | R2 : {r2:.4f}")

print("\n=======================================================")
print("✅ SELESAI! Seluruh 15 sub-kategori berhasil di-track ke MLflow!")
print("=======================================================")

2026/06/06 14:25:23 INFO mlflow.tracking.fluent: Experiment with name 'Weekly_SubCategory_Manual_Modeling' does not exist. Creating a new experiment.


🚀 Memulai otomatisasi training untuk 15 Sub-Kategori...

-------------------------------------------------------
[PROCESSING] Sub-Kategori: Accessories menggunakan GBR
✨ Weekly_Accessories_GBR Berhasil Tercatat!
   |-- Data Train/Test : 164 / 42 baris
   |-- MAE  : 8.0735 | RMSE : 11.7608 | R2 : 0.3098

-------------------------------------------------------
[PROCESSING] Sub-Kategori: Appliances menggunakan Random_Forest
✨ Weekly_Appliances_Random_Forest Berhasil Tercatat!
   |-- Data Train/Test : 164 / 41 baris
   |-- MAE  : 4.8383 | RMSE : 6.2032 | R2 : -0.0939

-------------------------------------------------------
[PROCESSING] Sub-Kategori: Art menggunakan GBR
✨ Weekly_Art_GBR Berhasil Tercatat!
   |-- Data Train/Test : 164 / 41 baris
   |-- MAE  : 7.1548 | RMSE : 8.9445 | R2 : 0.1083

-------------------------------------------------------
[PROCESSING] Sub-Kategori: Binders menggunakan Elastic_Net
✨ Weekly_Binders_Elastic_Net Berhasil Tercatat!
   |-- Data Train/Test : 164 / 42 b

In [4]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import optuna
import logging
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Impor semua algoritma
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import ElasticNet, Lars

# Matikan log bising dari Optuna agar terminal notebook tetap rapi
optuna.logging.set_verbosity(optuna.logging.WARNING)

# =========================================================================
# 1. PREPARASI DATA & METRIK CUSTOM
# =========================================================================
df_master = pd.read_csv(r'D:/Sem 4/PBL/data/processed/data_proses_mingguan.csv') 

def calculate_mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) if np.sum(mask) > 0 else 0.0

# Mapping Top 1 Model sesuai hasil PyCaret kamu kemarin
top_models_mapping = {
    "Accessories": "GBR",
    "Appliances":  "Random_Forest",
    "Art":         "GBR",
    "Binders":     "Elastic_Net",
    "Bookcases":   "Extra_Trees",
    "Chairs":      "Extra_Trees",
    "Envelopes":   "GBR",
    "Fasteners":   "Random_Forest",
    "Furnishings": "Random_Forest",
    "Labels":      "Random_Forest",
    "Paper":       "Random_Forest",
    "Phones":      "Random_Forest",
    "Storage":     "Extra_Trees",
    "Supplies":    "Lars",
    "Tables":      "Random_Forest"
}

# =========================================================================
# 2. SETTING EXPERIMENT MLFLOW BARU
# =========================================================================
mlflow.set_tracking_uri("file:///D:/Sem 4/PBL/model-pycaret/mlruns")
mlflow.set_experiment("Weekly_SubCategory_Optuna_Tuning")

print("🚀 Memulai Otomatisasi Hyperparameter Tuning dengan Optuna untuk 15 Sub-Kategori...")

# =========================================================================
# 3. LOOPING TUNING PIPELINE
# =========================================================================
for sub_cat, model_name in top_models_mapping.items():
    print(f"\n-------------------------------------------------------")
    print(f"[TUNING] Sub-Kategori: {sub_cat} | Model Base: {model_name}")
    
    # Filter data per sub-kategori
    df_sub = df_master[df_master['Sub-Category'] == sub_cat].copy()
    if df_sub.shape[0] < 10:
        continue
        
    # Pisahkan Fitur & Target (Sama persis dengan split baseline sebelumnya)
    X = df_sub.drop(columns=['Order Date', 'Sales', 'Profit', 'Quantity', 'Sub-Category'], errors='ignore')
    y = df_sub['Quantity']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Define Objective Function Dinamis untuk Optuna berdasarkan tipe Model
    def objective(trial):
        if model_name == "GBR":
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 250),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
                'max_depth': trial.suggest_int('max_depth', 3, 7),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
                'random_state': 42
            }
            model = GradientBoostingRegressor(**params)
            
        elif model_name == "Random_Forest":
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 3, 12),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 12),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 8),
                'max_features': trial.suggest_float('max_features', 0.5, 1.0),
                'random_state': 42
            }
            model = RandomForestRegressor(**params)
            
        elif model_name == "Extra_Trees":
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 3, 12),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 12),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 8),
                'random_state': 42
            }
            model = ExtraTreesRegressor(**params)
            
        elif model_name == "Elastic_Net":
            params = {
                'alpha': trial.suggest_float('alpha', 1e-4, 10.0, log=True),
                'l1_ratio': trial.suggest_float('l1_ratio', 0.0, 1.0),
                'random_state': 42
            }
            model = ElasticNet(**params)
            
        elif model_name == "Lars":
            params = {
                'eps': trial.suggest_float('eps', 1e-10, 1e-3, log=True)
            }
            model = Lars(**params)
            
        # Train sementara pada data train untuk dievaluasi oleh Optuna via data test
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        
        # Target Optuna: Meminimalkan nilai MAE (Bisa juga RMSE)
        mae_score = mean_absolute_error(y_test, preds)
        return mae_score

    # Jalankan Studi Optuna untuk mencari parameter terbaik (25 trials per sub-kategori agar cepat)
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=25)
    
    # Ambil Parameter Terbaik hasil temuan Optuna
    best_params = study.best_params
    
    # Build Ulang Model Menggunakan Parameter Terbaik Terbaik tersebut
    if model_name == "GBR":
        best_model = GradientBoostingRegressor(**best_params, random_state=42)
    elif model_name == "Random_Forest":
        best_model = RandomForestRegressor(**best_params, random_state=42)
    elif model_name == "Extra_Trees":
        best_model = ExtraTreesRegressor(**best_params, random_state=42)
    elif model_name == "Elastic_Net":
        best_model = ElasticNet(**best_params, random_state=42)
    elif model_name == "Lars":
        best_model = Lars(**best_params)
        
    # Latih model final hasil tuning pada data training
    best_model.fit(X_train, y_train)
    y_pred_tuned = best_model.predict(X_test)
    
    # Hitung Metrik Evaluasi Final Setelah Di-Tuning
    mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
    rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
    r2_tuned = r2_score(y_test, y_pred_tuned)
    mape_tuned = calculate_mape(y_test, y_pred_tuned)
    
    # Log Hasil Eksperimen Berharga Ini ke MLflow
    run_identity = f"Tuned_Weekly_{sub_cat}_{model_name}"
    with mlflow.start_run(run_name=run_identity):
        mlflow.log_param("sub_category", sub_cat)
        mlflow.log_param("tuning_framework", "Optuna")
        mlflow.log_params(best_params)
        
        mlflow.log_metric("MAE", mae_tuned)
        mlflow.log_metric("RMSE", rmse_tuned)
        mlflow.log_metric("R2", r2_tuned)
        mlflow.log_metric("MAPE", mape_tuned)
        
        # Daftarkan langsung model hasil tuning ini ke MLflow Registry
        mlflow.sklearn.log_model(best_model, artifact_path=f"tuned_model_{sub_cat.lower()}")
        
    # Tampilkan Hasil Perubahan Performa di Konsol Notebook
    print(f"✨ {run_identity} Sukses Di-Tuning & Masuk MLflow!")
    print(f"   |-- Best Params: {best_params}")
    print(f"   |-- [FINAL SCORE] MAE: {mae_tuned:.4f} | RMSE: {rmse_tuned:.4f} | R2: {r2_tuned:.4f}")

print("\n=======================================================")
print("✅ SELESAI! Semua model sub-kategori mingguan selesai di-tuning!")
print("=======================================================")

2026/06/06 14:39:40 INFO mlflow.tracking.fluent: Experiment with name 'Weekly_SubCategory_Optuna_Tuning' does not exist. Creating a new experiment.


🚀 Memulai Otomatisasi Hyperparameter Tuning dengan Optuna untuk 15 Sub-Kategori...

-------------------------------------------------------
[TUNING] Sub-Kategori: Accessories | Model Base: GBR
✨ Tuned_Weekly_Accessories_GBR Sukses Di-Tuning & Masuk MLflow!
   |-- Best Params: {'n_estimators': 249, 'learning_rate': 0.09514482427071258, 'max_depth': 7, 'subsample': 0.9907014642260804, 'min_samples_split': 2}
   |-- [FINAL SCORE] MAE: 7.4353 | RMSE: 11.2743 | R2: 0.3657

-------------------------------------------------------
[TUNING] Sub-Kategori: Appliances | Model Base: Random_Forest
✨ Tuned_Weekly_Appliances_Random_Forest Sukses Di-Tuning & Masuk MLflow!
   |-- Best Params: {'n_estimators': 215, 'max_depth': 3, 'min_samples_split': 6, 'min_samples_leaf': 4, 'max_features': 0.5597344906985203}
   |-- [FINAL SCORE] MAE: 4.3407 | RMSE: 5.5338 | R2: 0.1295

-------------------------------------------------------
[TUNING] Sub-Kategori: Art | Model Base: GBR
✨ Tuned_Weekly_Art_GBR Sukses Di

d:\Sem 4\PBL\venv\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.665e+04, tolerance: 5.215e+00
  model = cd_fast.enet_coordinate_descent(
d:\Sem 4\PBL\venv\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.789e+04, tolerance: 5.215e+00
  model = cd_fast.enet_coordinate_descent(
d:\Sem 4\PBL\venv\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.347e+04, tolerance: 5.215e+00
  model 

✨ Tuned_Weekly_Binders_Elastic_Net Sukses Di-Tuning & Masuk MLflow!
   |-- Best Params: {'alpha': 0.4463715072291181, 'l1_ratio': 0.9967939838751094}
   |-- [FINAL SCORE] MAE: 10.2395 | RMSE: 14.4000 | R2: 0.1295

-------------------------------------------------------
[TUNING] Sub-Kategori: Bookcases | Model Base: Extra_Trees
✨ Tuned_Weekly_Bookcases_Extra_Trees Sukses Di-Tuning & Masuk MLflow!
   |-- Best Params: {'n_estimators': 196, 'max_depth': 10, 'min_samples_split': 4, 'min_samples_leaf': 2}
   |-- [FINAL SCORE] MAE: 2.9386 | RMSE: 4.5088 | R2: 0.3035

-------------------------------------------------------
[TUNING] Sub-Kategori: Chairs | Model Base: Extra_Trees
✨ Tuned_Weekly_Chairs_Extra_Trees Sukses Di-Tuning & Masuk MLflow!
   |-- Best Params: {'n_estimators': 296, 'max_depth': 4, 'min_samples_split': 4, 'min_samples_leaf': 5}
   |-- [FINAL SCORE] MAE: 5.3868 | RMSE: 6.7906 | R2: 0.0298

-------------------------------------------------------
[TUNING] Sub-Kategori: Envelope

# finalisasi 14 sub kategori produk

In [8]:
import os
import pickle
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import ElasticNet

# 1. Buat folder penyimpan jika belum ada
os.makedirs(r'D:/Sem 4/PBL/model manual/models', exist_ok=True)

# 2. Definisikan Dictionary berisi Best Params hasil Optuna kamu tadi
best_tuned_configs = {
    "Accessories": GradientBoostingRegressor(n_estimators=249, learning_rate=0.0951, max_depth=7, subsample=0.9907, min_samples_split=2, random_state=42),
    "Appliances":  RandomForestRegressor(n_estimators=215, max_depth=3, min_samples_split=6, min_samples_leaf=4, max_features=0.5597, random_state=42),
    "Art":         GradientBoostingRegressor(n_estimators=122, learning_rate=0.0144, max_depth=3, subsample=0.7034, min_samples_split=2, random_state=42),
    "Binders":     ElasticNet(alpha=0.4463, l1_ratio=0.9967, max_iter=5000, random_state=42), # Ditambah max_iter agar konvergen
    "Bookcases":   ExtraTreesRegressor(n_estimators=196, max_depth=10, min_samples_split=4, min_samples_leaf=2, random_state=42),
    "Chairs":      ExtraTreesRegressor(n_estimators=296, max_depth=4, min_samples_split=4, min_samples_leaf=5, random_state=42),
    "Envelopes":   GradientBoostingRegressor(n_estimators=50, learning_rate=0.0682, max_depth=4, subsample=0.8429, min_samples_split=10, random_state=42),
    "Fasteners":   RandomForestRegressor(n_estimators=288, max_depth=6, min_samples_split=12, min_samples_leaf=3, max_features=0.8269, random_state=42),
    "Furnishings": RandomForestRegressor(n_estimators=275, max_depth=8, min_samples_split=11, min_samples_leaf=5, max_features=0.6827, random_state=42),
    "Labels":      RandomForestRegressor(n_estimators=197, max_depth=11, min_samples_split=6, min_samples_leaf=8, max_features=0.9552, random_state=42),
    "Paper":       RandomForestRegressor(n_estimators=69, max_depth=10, min_samples_split=6, min_samples_leaf=7, max_features=0.8967, random_state=42),
    "Phones":      RandomForestRegressor(n_estimators=296, max_depth=6, min_samples_split=11, min_samples_leaf=8, max_features=0.9367, random_state=42),
    "Storage":     ExtraTreesRegressor(n_estimators=138, max_depth=4, min_samples_split=11, min_samples_leaf=4, random_state=42),
    "Tables":      RandomForestRegressor(n_estimators=114, max_depth=8, min_samples_split=12, min_samples_leaf=5, max_features=0.9323, random_state=42),

}

# Load data master kembali
df_master = pd.read_csv(r'D:/Sem 4/PBL/data/processed/data_proses_mingguan.csv')

print("🛠️ Memulai proses Retrain ke 100% Data Historis...")

for sub_cat, model_obj in best_tuned_configs.items():
    # Filter data utuh tanpa split
    df_sub = df_master[df_master['Sub-Category'] == sub_cat].copy()
    
    X_full = df_sub.drop(columns=['Order Date', 'Sales', 'Profit', 'Quantity', 'Sub-Category'], errors='ignore')
    y_full = df_sub['Quantity']
    
    # Train ulang menggunakan seluruh data agar wawasan model optimal
    model_obj.fit(X_full, y_full)
    
    # Simpan secara lokal ke folder proyekmu
    file_path = f"models/model_{sub_cat.lower().replace(' ', '_')}.pkl"
    with open(file_path, 'wb') as f:
        pickle.dump(model_obj, f)
        
    print(f"✅ Model Final [{sub_cat}] sukses dilatih penuh & disimpan di: {file_path}")

print("\n🎉 Semua 14 Model siap diintegrasikan ke Backend Website (app.py)!")

🛠️ Memulai proses Retrain ke 100% Data Historis...
✅ Model Final [Accessories] sukses dilatih penuh & disimpan di: models/model_accessories.pkl
✅ Model Final [Appliances] sukses dilatih penuh & disimpan di: models/model_appliances.pkl
✅ Model Final [Art] sukses dilatih penuh & disimpan di: models/model_art.pkl
✅ Model Final [Binders] sukses dilatih penuh & disimpan di: models/model_binders.pkl
✅ Model Final [Bookcases] sukses dilatih penuh & disimpan di: models/model_bookcases.pkl
✅ Model Final [Chairs] sukses dilatih penuh & disimpan di: models/model_chairs.pkl
✅ Model Final [Envelopes] sukses dilatih penuh & disimpan di: models/model_envelopes.pkl
✅ Model Final [Fasteners] sukses dilatih penuh & disimpan di: models/model_fasteners.pkl
✅ Model Final [Furnishings] sukses dilatih penuh & disimpan di: models/model_furnishings.pkl
✅ Model Final [Labels] sukses dilatih penuh & disimpan di: models/model_labels.pkl
✅ Model Final [Paper] sukses dilatih penuh & disimpan di: models/model_paper.

# perbaikan untuk sub kategori supplier

In [5]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import ExtraTreesRegressor

# Matikan log bising dari Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# =========================================================================
# 1. PREPARASI DATA SPECIFIC "SUPPLIES"
# =========================================================================
df_master = pd.read_csv(r'D:/Sem 4/PBL/data/processed/data_proses_mingguan.csv')
df_supplies = df_master[df_master['Sub-Category'] == 'Supplies'].copy()

X = df_supplies.drop(columns=['Order Date', 'Sales', 'Profit', 'Quantity', 'Sub-Category'], errors='ignore')
y = df_supplies['Quantity']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def calculate_mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) if np.sum(mask) > 0 else 0.0

# Set URI MLflow kamu
mlflow.set_tracking_uri("file:///D:/Sem 4/PBL/model-pycaret/mlruns")

print(f"🧪 Memulai Eksperimen Penyelamatan untuk Sub-Kategori: Supplies")
print(f"   Jumlah data Train/Test: {X_train.shape[0]} / {X_test.shape[0]} baris\n")

# =========================================================================
# TAHAP 1: BASELINE EXTRA TREES FOR SUPPLIES
# =========================================================================
mlflow.set_experiment("Weekly_SubCategory_Manual_Modeling")

with mlflow.start_run(run_name="Weekly_Supplies_ExtraTrees_Baseline"):
    model_base = ExtraTreesRegressor(random_state=42)
    model_base.fit(X_train, y_train)
    preds_base = model_base.predict(X_test)
    
    mae_b = mean_absolute_error(y_test, preds_base)
    rmse_b = np.sqrt(mean_squared_error(y_test, preds_base))
    r2_b = r2_score(y_test, preds_base)
    mape_b = calculate_mape(y_test, preds_base)
    
    mlflow.log_param("sub_category", "Supplies")
    mlflow.log_params(model_base.get_params())
    mlflow.log_metric("MAE", mae_b)
    mlflow.log_metric("RMSE", rmse_b)
    mlflow.log_metric("R2", r2_b)
    mlflow.log_metric("MAPE", mape_b)
    
    print(f"📊 [STAGE 1 - BASELINE] Extra Trees Supplies Selesai!")
    print(f"   |-- MAE: {mae_b:.4f} | RMSE: {rmse_b:.4f} | R2: {r2_b:.4f}")

# =========================================================================
# TAHAP 2: OPTUNA TUNING EXTRA TREES FOR SUPPLIES
# =========================================================================
mlflow.set_experiment("Weekly_SubCategory_Optuna_Tuning")

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 12),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 8),
        'random_state': 42
    }
    model = ExtraTreesRegressor(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return mean_absolute_error(y_test, preds)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30) # 30 percobaan pencarian
best_params = study.best_params

with mlflow.start_run(run_name="Tuned_Weekly_Supplies_ExtraTrees"):
    model_tuned = ExtraTreesRegressor(**best_params, random_state=42)
    model_tuned.fit(X_train, y_train)
    preds_tuned = model_tuned.predict(X_test)
    
    mae_t = mean_absolute_error(y_test, preds_tuned)
    rmse_t = np.sqrt(mean_squared_error(y_test, preds_tuned))
    r2_t = r2_score(y_test, preds_tuned)
    mape_t = calculate_mape(y_test, preds_tuned)
    
    mlflow.log_param("sub_category", "Supplies")
    mlflow.log_param("tuning_framework", "Optuna")
    mlflow.log_params(best_params)
    mlflow.log_metric("MAE", mae_t)
    mlflow.log_metric("RMSE", rmse_t)
    mlflow.log_metric("R2", r2_t)
    mlflow.log_metric("MAPE", mape_t)
    
    # Registrasikan model hasil penyelamatan ini ke MLflow Artifact
    mlflow.sklearn.log_model(model_tuned, artifact_path="tuned_model_supplies_et")
    
    print(f"\n🚀 [STAGE 2 - TUNED] Extra Trees Supplies Selesai!")
    print(f"   |-- Best Params: {best_params}")
    print(f"   |-- MAE: {mae_t:.4f} | RMSE: {rmse_t:.4f} | R2: {r2_t:.4f}")

print("\n=======================================================")
print("✅ Selesai! Semua rekam jejak eksperimen Supplies masuk ke MLflow.")
print("=======================================================")

🧪 Memulai Eksperimen Penyelamatan untuk Sub-Kategori: Supplies
   Jumlah data Train/Test: 159 / 40 baris

📊 [STAGE 1 - BASELINE] Extra Trees Supplies Selesai!
   |-- MAE: 2.7820 | RMSE: 4.1088 | R2: 0.1174

🚀 [STAGE 2 - TUNED] Extra Trees Supplies Selesai!
   |-- Best Params: {'n_estimators': 220, 'max_depth': 10, 'min_samples_split': 8, 'min_samples_leaf': 2}
   |-- MAE: 2.8693 | RMSE: 4.1029 | R2: 0.1199

✅ Selesai! Semua rekam jejak eksperimen Supplies masuk ke MLflow.


# finalisasi model untuk sub kategori suplies

In [ ]:
import os
import pickle
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor

# =========================================================================
# 1. PREPARASI DIREKTORI
# =========================================================================
# Memastikan folder 'models_weekly' sudah terbentuk
os.makedirs(r'D:/Sem 4/PBL/model manual/models', exist_ok=True)

# =========================================================================
# 2. INISIALISASI MODEL DENGAN BEST PARAMS OPTUNA
# =========================================================================
best_supplies_model = ExtraTreesRegressor(
    n_estimators=220,
    max_depth=10,
    min_samples_split=8,
    min_samples_leaf=2,
    random_state=42
)

# =========================================================================
# 3. LOAD & FILTER DATA MASTER UTUH
# =========================================================================
df_master = pd.read_csv(r'D:/Sem 4/PBL/data/processed/data_proses_mingguan.csv')
df_supplies = df_master[df_master['Sub-Category'] == 'Supplies'].copy()

# Pisahkan Fitur (X) dan Target (y)
X_full = df_supplies.drop(columns=['Order Date', 'Sales', 'Profit', 'Quantity', 'Sub-Category'], errors='ignore')
y_full = df_supplies['Quantity']

# =========================================================================
# 4. RETRAIN & SAVE MODEL
# =========================================================================
print("🛠️ Memulai proses Retrain ke 100% Data Historis Supplies...")

# Latih model menggunakan seluruh data Supplies agar pola yang diserap maksimal
best_supplies_model.fit(X_full, y_full)

# Tentukan path file penyimpanan .pkl
file_path = r'D:/Sem 4/PBL/model manual/models/model_weekly_supplies.pkl'

# Simpan model menggunakan pickle
with open(file_path, 'wb') as f:
    pickle.dump(best_supplies_model, f)

print(f"✅ Model Final Extra Trees [Supplies] sukses dilatih penuh!")
print(f"📦 File model disimpan di: {file_path}")
print("=======================================================")

🛠️ Memulai proses Retrain ke 100% Data Historis Supplies...
✅ Model Final Extra Trees [Supplies] sukses dilatih penuh!
📦 File model disimpan di: D:/Sem 4/PBL/model manual/models/model_weekly_supplies.pkl


# penerapan learning curve

In [2]:
import os
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Impor semua algoritma yang dibutuhkan berdasarkan tabel PyCaret-mu
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor, AdaBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import ElasticNet, Lars, Lasso, LinearRegression

# =========================================================================
# 1. PREPARASI DATA MASTER MINGGUAN
# =========================================================================
df_master = pd.read_csv(r'D:/Sem 4/PBL/data/processed/data_proses_mingguan.csv')

# Fungsi evaluasi custom MAPE
def calculate_mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) if np.sum(mask) > 0 else 0.0

# =========================================================================
# 2. DICTIONARY MAPPING (Kunci Otomatisasi)
# =========================================================================
top_models_mapping = {
    "Accessories": ("GBR", GradientBoostingRegressor(random_state=42)),
    "Appliances":  ("Random_Forest", RandomForestRegressor(random_state=42)),
    "Art":         ("GBR", GradientBoostingRegressor(random_state=42)),
    "Binders":     ("Elastic_Net", ElasticNet(random_state=42)),
    "Bookcases":   ("Extra_Trees", ExtraTreesRegressor(random_state=42)),
    "Chairs":      ("Extra_Trees", ExtraTreesRegressor(random_state=42)),
    "Envelopes":   ("GBR", GradientBoostingRegressor(random_state=42)),
    "Fasteners":   ("Random_Forest", RandomForestRegressor(random_state=42)),
    "Furnishings": ("Random_Forest", RandomForestRegressor(random_state=42)),
    "Labels":      ("Random_Forest", RandomForestRegressor(random_state=42)),
    "Paper":       ("Random_Forest", RandomForestRegressor(random_state=42)),
    "Phones":      ("Random_Forest", RandomForestRegressor(random_state=42)),
    "Storage":     ("Extra_Trees", ExtraTreesRegressor(random_state=42)),
    "Supplies":    ("Lars", Lars()),
    "Tables":      ("Random_Forest", RandomForestRegressor(random_state=42))
}

# =========================================================================
# 3. SETTING MLFLOW EXPERIMENT
# =========================================================================
mlflow.set_tracking_uri("file:///D:/Sem 4/PBL/model-pycaret/mlruns")
mlflow.set_experiment("Weekly_SubCategory_Manual_Modeling")

print("🚀 Memulai otomatisasi training & analisis grafik untuk 15 Sub-Kategori...")

# =========================================================================
# 4. LOOPING AUTOMATION PIPELINE
# =========================================================================
for sub_cat, (model_name, model_obj) in top_models_mapping.items():
    print(f"\n-------------------------------------------------------")
    print(f"[PROCESSING] Sub-Kategori: {sub_cat} menggunakan {model_name}")
    
    # a. Filter data spesifik untuk sub-kategori saat ini
    df_sub = df_master[df_master['Sub-Category'] == sub_cat].copy()
    
    # Batas aman minimum data untuk melakukan split dan 3-Fold Learning Curve
    if df_sub.shape[0] < 15:
        print(f"⚠️ Data untuk {sub_cat} terlalu sedikit ({df_sub.shape[0]} baris), dilewati otomatis.")
        continue
        
    # b. Pisahkan Fitur dan Target
    X = df_sub.drop(columns=['Order Date', 'Sales', 'Profit', 'Quantity', 'Sub-Category'], errors='ignore')
    y = df_sub['Quantity']
    
    # c. Split Data (80:20)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # d. Jalankan Tracking MLflow
    run_identity = f"Weekly_{sub_cat}_{model_name}"
    with mlflow.start_run(run_name=run_identity):
        
        # Train Model Utama
        model_obj.fit(X_train, y_train)
        
        # Prediksi
        y_pred = model_obj.predict(X_test)
        
        # Hitung Metrik Penilaian
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        mape = calculate_mape(y_test, y_pred)
        
        # -----------------------------------------------------------------
        # PENYISIPAN LOGIKA PEMBUATAN LEARNING CURVE
        # -----------------------------------------------------------------
        plot_filename = f"weekly_{sub_cat.lower()}_learning_curve.png"
        has_plot = False
        
        try:
            # Gunakan cv=3 karena beberapa sub-kategori memiliki jumlah baris data yang ketat
            train_sizes, train_scores, valid_scores = learning_curve(
                model_obj, X_train, y_train, cv=3, scoring='r2', 
                train_sizes=np.linspace(0.1, 1.0, 5), n_jobs=-1, random_state=42
            )
            
            # Plotting Grafik
            plt.figure(figsize=(8, 4))
            plt.plot(train_sizes, np.mean(train_scores, axis=1), 'o-', color="red", label="Train Score ($R^2$)")
            plt.plot(train_sizes, np.mean(valid_scores, axis=1), 'o-', color="green", label="CV Score ($R^2$)")
            plt.title(f"Learning Curve: {sub_cat} ({model_name})")
            plt.xlabel("Training Samples")
            plt.ylabel("Score $R^2$")
            plt.legend(loc="best")
            plt.grid(True)
            
            # Simpan secara lokal
            plt.savefig(plot_filename, bbox_inches='tight')
            plt.close()
            has_plot = True
        except Exception as e:
            print(f"⚠️ Gagal membuat Learning Curve untuk {sub_cat}: {str(e)}")
            plt.close()

        # -----------------------------------------------------------------
        # LOGGING KE SERVER MLFLOW
        # -----------------------------------------------------------------
        # 1. Log parameter dasar
        mlflow.log_param("sub_category", sub_cat)
        mlflow.log_params(model_obj.get_params())
        
        # 2. Log seluruh metrik evaluasi
        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("RMSE", rmse)
        mlflow.log_metric("R2", r2)
        mlflow.log_metric("MAPE", mape)
        
        # 3. Kirim file gambar plot ke folder artifact MLflow jika berhasil dibuat
        if has_plot:
            mlflow.log_artifact(plot_filename, artifact_path="evaluation_plots")
            if os.path.exists(plot_filename):
                os.remove(plot_filename) # Hapus file sampah lokal
        
        # 4. Registrasikan file model .pkl
        mlflow.sklearn.log_model(model_obj, artifact_path=f"model_{sub_cat.lower()}")
        
        # Cetak konfirmasi instan di terminal
        print(f"✨ {run_identity} Berhasil Tercatat!")
        print(f"   |-- Data Train/Test : {X_train.shape[0]} / {X_test.shape[0]} baris")
        print(f"   |-- MAE  : {mae:.4f} | RMSE : {rmse:.4f} | R2 : {r2:.4f}")
        if has_plot:
            print(f"   |-- [ARTIFACT] Grafik Learning Curve sukses diunggah!")

print("\n=======================================================")
print("✅ SELESAI! Seluruh 15 sub-kategori & Learning Curves sukses terekam!")
print("=======================================================")

🚀 Memulai otomatisasi training & analisis grafik untuk 15 Sub-Kategori...

-------------------------------------------------------
[PROCESSING] Sub-Kategori: Accessories menggunakan GBR
✨ Weekly_Accessories_GBR Berhasil Tercatat!
   |-- Data Train/Test : 164 / 42 baris
   |-- MAE  : 8.0735 | RMSE : 11.7608 | R2 : 0.3098
   |-- [ARTIFACT] Grafik Learning Curve sukses diunggah!

-------------------------------------------------------
[PROCESSING] Sub-Kategori: Appliances menggunakan Random_Forest
✨ Weekly_Appliances_Random_Forest Berhasil Tercatat!
   |-- Data Train/Test : 164 / 41 baris
   |-- MAE  : 4.8383 | RMSE : 6.2032 | R2 : -0.0939
   |-- [ARTIFACT] Grafik Learning Curve sukses diunggah!

-------------------------------------------------------
[PROCESSING] Sub-Kategori: Art menggunakan GBR
✨ Weekly_Art_GBR Berhasil Tercatat!
   |-- Data Train/Test : 164 / 41 baris
   |-- MAE  : 7.1548 | RMSE : 8.9445 | R2 : 0.1083
   |-- [ARTIFACT] Grafik Learning Curve sukses diunggah!

---------

In [4]:
import os
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import optuna
import logging
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Impor semua algoritma
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import ElasticNet, Lars

# Matikan log bising dari Optuna agar terminal notebook tetap rapi
optuna.logging.set_verbosity(optuna.logging.WARNING)

# =========================================================================
# 1. PREPARASI DATA & METRIK CUSTOM
# =========================================================================
df_master = pd.read_csv(r'D:/Sem 4/PBL/data/processed/data_proses_mingguan.csv') 

def calculate_mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) if np.sum(mask) > 0 else 0.0

# Mapping Top 1 Model sesuai hasil PyCaret kamu kemarin
top_models_mapping = {
    "Accessories": "GBR",
    "Appliances":  "Random_Forest",
    "Art":         "GBR",
    "Binders":     "Elastic_Net",
    "Bookcases":   "Extra_Trees",
    "Chairs":      "Extra_Trees",
    "Envelopes":   "GBR",
    "Fasteners":   "Random_Forest",
    "Furnishings": "Random_Forest",
    "Labels":      "Random_Forest",
    "Paper":       ("Random_Forest"),
    "Phones":      "Random_Forest",
    "Storage":     "Extra_Trees",
    "Supplies":    "Lars",
    "Tables":      "Random_Forest"
}

# =========================================================================
# 2. SETTING EXPERIMENT MLFLOW BARU
# =========================================================================
mlflow.set_tracking_uri("file:///D:/Sem 4/PBL/model-pycaret/mlruns")
mlflow.set_experiment("Weekly_SubCategory_Optuna_Tuning")

print("🚀 Memulai Otomatisasi Hyperparameter Tuning & Learning Curve untuk 15 Sub-Kategori...")

# =========================================================================
# 3. LOOPING TUNING PIPELINE
# =========================================================================
for sub_cat, model_name in top_models_mapping.items():
    print(f"\n-------------------------------------------------------")
    print(f"[TUNING] Sub-Kategori: {sub_cat} | Model Base: {model_name}")
    
    # Filter data per sub-kategori
    df_sub = df_master[df_master['Sub-Category'] == sub_cat].copy()
    
    # Batas aman filter dinaikkan ke 15 agar proses cv=3 pada Learning Curve tidak crash
    if df_sub.shape[0] < 15:
        print(f"⚠️ Data untuk {sub_cat} terlalu sedikit ({df_sub.shape[0]} baris), dilewati.")
        continue
        
    # Pisahkan Fitur & Target
    X = df_sub.drop(columns=['Order Date', 'Sales', 'Profit', 'Quantity', 'Sub-Category'], errors='ignore')
    y = df_sub['Quantity']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Define Objective Function Dinamis untuk Optuna berdasarkan tipe Model
    def objective(trial):
        if model_name == "GBR":
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 250),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
                'max_depth': trial.suggest_int('max_depth', 3, 7),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
                'random_state': 42
            }
            model = GradientBoostingRegressor(**params)
            
        elif model_name == "Random_Forest":
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 3, 12),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 12),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 8),
                'max_features': trial.suggest_float('max_features', 0.5, 1.0),
                'random_state': 42
            }
            model = RandomForestRegressor(**params)
            
        elif model_name == "Extra_Trees":
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 3, 12),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 12),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 8),
                'random_state': 42
            }
            model = ExtraTreesRegressor(**params)
            
        elif model_name == "Elastic_Net":
            params = {
                'alpha': trial.suggest_float('alpha', 1e-4, 10.0, log=True),
                'l1_ratio': trial.suggest_float('l1_ratio', 0.0, 1.0),
                'random_state': 42
            }
            model = ElasticNet(**params)
            
        elif model_name == "Lars":
            params = {
                'eps': trial.suggest_float('eps', 1e-10, 1e-3, log=True)
            }
            model = Lars(**params)
            
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        return mean_absolute_error(y_test, preds)

    # Jalankan Studi Optuna (25 trials)
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=25)
    best_params = study.best_params
    
    # Build Ulang Model Menggunakan Parameter Terbaik
    if model_name == "GBR":
        best_model = GradientBoostingRegressor(**best_params, random_state=42)
    elif model_name == "Random_Forest":
        best_model = RandomForestRegressor(**best_params, random_state=42)
    elif model_name == "Extra_Trees":
        best_model = ExtraTreesRegressor(**best_params, random_state=42)
    elif model_name == "Elastic_Net":
        best_model = ElasticNet(**best_params, random_state=42)
    elif model_name == "Lars":
        best_model = Lars(**best_params)
        
    # Latih model final hasil tuning
    best_model.fit(X_train, y_train)
    y_pred_tuned = best_model.predict(X_test)
    
    # Hitung Metrik Evaluasi Final
    mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
    rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
    r2_tuned = r2_score(y_test, y_pred_tuned)
    mape_tuned = calculate_mape(y_test, y_pred_tuned)
    
    # Log Hasil Eksperimen ke MLflow
    run_identity = f"Tuned_Weekly_{sub_cat}_{model_name}"
    with mlflow.start_run(run_name=run_identity):
        
        # -----------------------------------------------------------------
        # PROSES GENERASI LEARNING CURVE UNTUK MODEL TERBAIK
        # -----------------------------------------------------------------
        plot_filename = f"tuned_weekly_{sub_cat.lower()}_learning_curve.png"
        has_plot = False
        
        try:
            train_sizes, train_scores, valid_scores = learning_curve(
                best_model, X_train, y_train, cv=3, scoring='r2', 
                train_sizes=np.linspace(0.1, 1.0, 5), n_jobs=-1, random_state=42
            )
            
            plt.figure(figsize=(8, 4))
            plt.plot(train_sizes, np.mean(train_scores, axis=1), 'o-', color="red", label="Train Score ($R^2$)")
            plt.plot(train_sizes, np.mean(valid_scores, axis=1), 'o-', color="green", label="CV Score ($R^2$)")
            plt.title(f"Learning Curve: Tuned {sub_cat} ({model_name})")
            plt.xlabel("Training Samples")
            plt.ylabel("Score $R^2$")
            plt.legend(loc="best")
            plt.grid(True)
            
            plt.savefig(plot_filename, bbox_inches='tight')
            plt.close()
            has_plot = True
        except Exception as e:
            print(f"⚠️ Grafik {sub_cat} dilewati karena batasan internal: {str(e)}")
            plt.close()

        # -----------------------------------------------------------------
        # TRACKING ARTIFACTS & METRICS KE SERVER MLFLOW
        # -----------------------------------------------------------------
        mlflow.log_param("sub_category", sub_cat)
        mlflow.log_param("tuning_framework", "Optuna")
        mlflow.log_params(best_params)
        
        mlflow.log_metric("MAE", mae_tuned)
        mlflow.log_metric("RMSE", rmse_tuned)
        mlflow.log_metric("R2", r2_tuned)
        mlflow.log_metric("MAPE", mape_tuned)
        
        # Unggah plot gambar ke folder artifact MLflow jika berhasil dibentuk
        if has_plot:
            mlflow.log_artifact(plot_filename, artifact_path="evaluation_plots")
            if os.path.exists(plot_filename):
                os.remove(plot_filename) # Hapus file lokal sementara
        
        # Daftarkan model biner .pkl
        mlflow.sklearn.log_model(best_model, artifact_path=f"tuned_model_{sub_cat.lower()}")
        
    # Tampilkan Hasil di Konsol
    print(f"✨ {run_identity} Sukses Di-Tuning & Masuk MLflow!")
    print(f"   |-- Best Params: {best_params}")
    print(f"   |-- [FINAL SCORE] MAE: {mae_tuned:.4f} | RMSE: {rmse_tuned:.4f} | R2: {r2_tuned:.4f}")
    if has_plot:
        print(f"   |-- [ARTIFACT] Grafik Learning Curve sukses diunggah!")

print("\n=======================================================")
print("✅ SELESAI! Semua model sub-kategori mingguan selesai di-tuning & di-plot!")
print("=======================================================")

🚀 Memulai Otomatisasi Hyperparameter Tuning & Learning Curve untuk 15 Sub-Kategori...

-------------------------------------------------------
[TUNING] Sub-Kategori: Accessories | Model Base: GBR
✨ Tuned_Weekly_Accessories_GBR Sukses Di-Tuning & Masuk MLflow!
   |-- Best Params: {'n_estimators': 121, 'learning_rate': 0.1411143168338475, 'max_depth': 7, 'subsample': 0.9111023559762274, 'min_samples_split': 2}
   |-- [FINAL SCORE] MAE: 7.0794 | RMSE: 10.7763 | R2: 0.4205
   |-- [ARTIFACT] Grafik Learning Curve sukses diunggah!

-------------------------------------------------------
[TUNING] Sub-Kategori: Appliances | Model Base: Random_Forest
✨ Tuned_Weekly_Appliances_Random_Forest Sukses Di-Tuning & Masuk MLflow!
   |-- Best Params: {'n_estimators': 239, 'max_depth': 3, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 0.6704401436257208}
   |-- [FINAL SCORE] MAE: 4.3735 | RMSE: 5.5438 | R2: 0.1263
   |-- [ARTIFACT] Grafik Learning Curve sukses diunggah!

-----------------

d:\Sem 4\PBL\venv\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.473e+04, tolerance: 5.215e+00
  model = cd_fast.enet_coordinate_descent(
d:\Sem 4\PBL\venv\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.935e+01, tolerance: 5.215e+00
  model = cd_fast.enet_coordinate_descent(
d:\Sem 4\PBL\venv\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.716e+04, tolerance: 5.215e+00
  model 

✨ Tuned_Weekly_Binders_Elastic_Net Sukses Di-Tuning & Masuk MLflow!
   |-- Best Params: {'alpha': 0.5649816214396783, 'l1_ratio': 0.9765175697233092}
   |-- [FINAL SCORE] MAE: 10.2410 | RMSE: 14.3618 | R2: 0.1341
   |-- [ARTIFACT] Grafik Learning Curve sukses diunggah!

-------------------------------------------------------
[TUNING] Sub-Kategori: Bookcases | Model Base: Extra_Trees
✨ Tuned_Weekly_Bookcases_Extra_Trees Sukses Di-Tuning & Masuk MLflow!
   |-- Best Params: {'n_estimators': 124, 'max_depth': 10, 'min_samples_split': 10, 'min_samples_leaf': 2}
   |-- [FINAL SCORE] MAE: 2.9816 | RMSE: 4.5293 | R2: 0.2972
   |-- [ARTIFACT] Grafik Learning Curve sukses diunggah!

-------------------------------------------------------
[TUNING] Sub-Kategori: Chairs | Model Base: Extra_Trees
✨ Tuned_Weekly_Chairs_Extra_Trees Sukses Di-Tuning & Masuk MLflow!
   |-- Best Params: {'n_estimators': 136, 'max_depth': 4, 'min_samples_split': 9, 'min_samples_leaf': 2}
   |-- [FINAL SCORE] MAE: 5.3998 |

In [3]:
import os
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import optuna
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import ExtraTreesRegressor

# Matikan log bising dari Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# =========================================================================
# 1. PREPARASI DATA SPECIFIC "SUPPLIES"
# =========================================================================
df_master = pd.read_csv(r'D:/Sem 4/PBL/data/processed/data_proses_mingguan.csv')
df_supplies = df_master[df_master['Sub-Category'] == 'Supplies'].copy()

X = df_supplies.drop(columns=['Order Date', 'Sales', 'Profit', 'Quantity', 'Sub-Category'], errors='ignore')
y = df_supplies['Quantity']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def calculate_mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) if np.sum(mask) > 0 else 0.0

# Set URI MLflow kamu
mlflow.set_tracking_uri("file:///D:/Sem 4/PBL/model-pycaret/mlruns")

print(f"🧪 Memulai Eksperimen Penyelamatan untuk Sub-Kategori: Supplies")
print(f"   Jumlah data Train/Test: {X_train.shape[0]} / {X_test.shape[0]} baris\n")


# =========================================================================
# TAHAP 1: BASELINE EXTRA TREES FOR SUPPLIES
# =========================================================================
mlflow.set_experiment("Weekly_SubCategory_Manual_Modeling")

with mlflow.start_run(run_name="Weekly_Supplies_ExtraTrees_Baseline"):
    model_base = ExtraTreesRegressor(random_state=42)
    model_base.fit(X_train, y_train)
    preds_base = model_base.predict(X_test)
    
    mae_b = mean_absolute_error(y_test, preds_base)
    rmse_b = np.sqrt(mean_squared_error(y_test, preds_base))
    r2_b = r2_score(y_test, preds_base)
    mape_b = calculate_mape(y_test, preds_base)
    
    # --- LEARNING CURVE BASELINE ---
    plot_base_name = "weekly_supplies_baseline_learning_curve.png"
    has_base_plot = False
    try:
        train_sizes_b, train_scores_b, valid_scores_b = learning_curve(
            model_base, X_train, y_train, cv=3, scoring='r2', 
            train_sizes=np.linspace(0.1, 1.0, 5), n_jobs=-1, random_state=42
        )
        plt.figure(figsize=(8, 4))
        plt.plot(train_sizes_b, np.mean(train_scores_b, axis=1), 'o-', color="red", label="Train Score (R2)")
        plt.plot(train_sizes_b, np.mean(valid_scores_b, axis=1), 'o-', color="green", label="CV Score (R2)")
        plt.title("Learning Curve: Supplies Extra Trees Baseline")
        plt.xlabel("Training Samples")
        plt.ylabel("R2 Score")
        plt.legend(loc="best")
        plt.grid(True)
        plt.savefig(plot_base_name, bbox_inches='tight')
        plt.close()
        has_base_plot = True
    except Exception as e:
        print(f"⚠️ Gagal membuat Learning Curve Baseline: {str(e)}")
        plt.close()

    # Log Komponen ke MLflow
    mlflow.log_param("sub_category", "Supplies")
    mlflow.log_params(model_base.get_params())
    mlflow.log_metric("MAE", mae_b)
    mlflow.log_metric("RMSE", rmse_b)
    mlflow.log_metric("R2", r2_b)
    mlflow.log_metric("MAPE", mape_b)
    
    if has_base_plot:
        mlflow.log_artifact(plot_base_name, artifact_path="evaluation_plots")
        if os.path.exists(plot_base_name):
            os.remove(plot_base_name)
    
    print(f"📊 [STAGE 1 - BASELINE] Extra Trees Supplies Selesai!")
    print(f"   |-- MAE: {mae_b:.4f} | RMSE: {rmse_b:.4f} | R2: {r2_b:.4f}")
    if has_base_plot:
        print(f"   |-- [ARTIFACT] Grafik Learning Curve Baseline sukses diunggah!")


# =========================================================================
# TAHAP 2: OPTUNA TUNING EXTRA TREES FOR SUPPLIES
# =========================================================================
mlflow.set_experiment("Weekly_SubCategory_Optuna_Tuning")

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 12),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 8),
        'random_state': 42
    }
    model = ExtraTreesRegressor(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return mean_absolute_error(y_test, preds)

print("\n-> Memulai Tuning Optuna untuk Extra Trees Supplies (30 Trials)...")
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30) 
best_params = study.best_params

with mlflow.start_run(run_name="Tuned_Weekly_Supplies_ExtraTrees"):
    model_tuned = ExtraTreesRegressor(**best_params, random_state=42)
    model_tuned.fit(X_train, y_train)
    preds_tuned = model_tuned.predict(X_test)
    
    mae_t = mean_absolute_error(y_test, preds_tuned)
    rmse_t = np.sqrt(mean_squared_error(y_test, preds_tuned))
    r2_t = r2_score(y_test, preds_tuned)
    mape_t = calculate_mape(y_test, preds_tuned)
    
    # --- LEARNING CURVE TUNED MODEL ---
    plot_tuned_name = "weekly_supplies_tuned_learning_curve.png"
    has_tuned_plot = False
    try:
        train_sizes_t, train_scores_t, valid_scores_t = learning_curve(
            model_tuned, X_train, y_train, cv=3, scoring='r2', 
            train_sizes=np.linspace(0.1, 1.0, 5), n_jobs=-1, random_state=42
        )
        plt.figure(figsize=(8, 4))
        plt.plot(train_sizes_t, np.mean(train_scores_t, axis=1), 'o-', color="red", label="Train Score (R2)")
        plt.plot(train_sizes_t, np.mean(valid_scores_t, axis=1), 'o-', color="green", label="CV Score (R2)")
        plt.title("Learning Curve: Supplies Extra Trees Tuned")
        plt.xlabel("Training Samples")
        plt.ylabel("R2 Score")
        plt.legend(loc="best")
        plt.grid(True)
        plt.savefig(plot_tuned_name, bbox_inches='tight')
        plt.close()
        has_tuned_plot = True
    except Exception as e:
        print(f"⚠️ Gagal membuat Learning Curve Tuned: {str(e)}")
        plt.close()

    # Log Komponen ke MLflow
    mlflow.log_param("sub_category", "Supplies")
    mlflow.log_param("tuning_framework", "Optuna")
    mlflow.log_params(best_params)
    mlflow.log_metric("MAE", mae_t)
    mlflow.log_metric("RMSE", rmse_t)
    mlflow.log_metric("R2", r2_t)
    mlflow.log_metric("MAPE", mape_t)
    
    if has_tuned_plot:
        mlflow.log_artifact(plot_tuned_name, artifact_path="evaluation_plots")
        if os.path.exists(plot_tuned_name):
            os.remove(plot_tuned_name)
            
    # Registrasikan model hasil penyelamatan ini ke MLflow Artifact
    mlflow.sklearn.log_model(model_tuned, artifact_path="tuned_model_supplies_et")
    
    print(f"\n🚀 [STAGE 2 - TUNED] Extra Trees Supplies Selesai!")
    print(f"   |-- Best Params: {best_params}")
    print(f"   |-- MAE: {mae_t:.4f} | RMSE: {rmse_t:.4f} | R2: {r2_t:.4f}")
    if has_tuned_plot:
        print(f"   |-- [ARTIFACT] Grafik Learning Curve Tuned sukses diunggah!")

print("\n=======================================================")
print("✅ Selesai! Semua rekam jejak eksperimen Supplies masuk ke MLflow.")
print("=======================================================")

🧪 Memulai Eksperimen Penyelamatan untuk Sub-Kategori: Supplies
   Jumlah data Train/Test: 159 / 40 baris

📊 [STAGE 1 - BASELINE] Extra Trees Supplies Selesai!
   |-- MAE: 2.7820 | RMSE: 4.1088 | R2: 0.1174
   |-- [ARTIFACT] Grafik Learning Curve Baseline sukses diunggah!

-> Memulai Tuning Optuna untuk Extra Trees Supplies (30 Trials)...

🚀 [STAGE 2 - TUNED] Extra Trees Supplies Selesai!
   |-- Best Params: {'n_estimators': 174, 'max_depth': 11, 'min_samples_split': 2, 'min_samples_leaf': 2}
   |-- MAE: 2.8064 | RMSE: 4.0709 | R2: 0.1336
   |-- [ARTIFACT] Grafik Learning Curve Tuned sukses diunggah!

✅ Selesai! Semua rekam jejak eksperimen Supplies masuk ke MLflow.


In [5]:
import os
import pickle
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor

# =========================================================================
# 1. PREPARASI DIREKTORI
# =========================================================================
# Memastikan folder 'models_weekly' sudah terbentuk
os.makedirs(r'D:/Sem 4/PBL/model manual/models/by coding', exist_ok=True)

# =========================================================================
# 2. INISIALISASI MODEL DENGAN BEST PARAMS OPTUNA
# =========================================================================
best_supplies_model = ExtraTreesRegressor(
    n_estimators=220,
    max_depth=10,
    min_samples_split=8,
    min_samples_leaf=2,
    random_state=42
)

# =========================================================================
# 3. LOAD & FILTER DATA MASTER UTUH
# =========================================================================
df_master = pd.read_csv(r'D:/Sem 4/PBL/data/processed/data_proses_mingguan.csv')
df_supplies = df_master[df_master['Sub-Category'] == 'Supplies'].copy()

# Pisahkan Fitur (X) dan Target (y)
X_full = df_supplies.drop(columns=['Order Date', 'Sales', 'Profit', 'Quantity', 'Sub-Category'], errors='ignore')
y_full = df_supplies['Quantity']

# =========================================================================
# 4. RETRAIN & SAVE MODEL
# =========================================================================
print("🛠️ Memulai proses Retrain ke 100% Data Historis Supplies...")

# Latih model menggunakan seluruh data Supplies agar pola yang diserap maksimal
best_supplies_model.fit(X_full, y_full)

# Tentukan path file penyimpanan .pkl
file_path = r'D:/Sem 4/PBL/model manual/models/by coding/model_weekly_supplies.pkl'

# Simpan model menggunakan pickle
with open(file_path, 'wb') as f:
    pickle.dump(best_supplies_model, f)

print(f"✅ Model Final Extra Trees [Supplies] sukses dilatih penuh!")
print(f"📦 File model disimpan di: {file_path}")
print("=======================================================")

🛠️ Memulai proses Retrain ke 100% Data Historis Supplies...
✅ Model Final Extra Trees [Supplies] sukses dilatih penuh!
📦 File model disimpan di: D:/Sem 4/PBL/model manual/models/by coding/model_weekly_supplies.pkl
